

1. **Genre / Studio Jaccard overlap**:  structural sanity check (no ratings needed).
2. **Held-out user ratings as relevance signal**: Precision@K / Recall@K / MAP@K using high ratings (>=8) as ground-truth "similar".

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

In [2]:
DATA_DIR = Path(os.getcwd()).parent / "data"
ARTIFACT_DIR = Path(os.getcwd()).parent / "faiss_artifacts"

INDEX_PATH = ARTIFACT_DIR / "content_faiss_index.bin"
MAL_TO_FAISS_PATH = ARTIFACT_DIR / "mal_id_to_faiss_id.json"
FAISS_TO_MAL_PATH = ARTIFACT_DIR / "faiss_id_to_mal_id.json"

In [3]:
index = faiss.read_index(str(INDEX_PATH))

with open(MAL_TO_FAISS_PATH, "r", encoding="utf-8") as f:
    mal_id_to_faiss_id = {int(k): int(v) for k, v in json.load(f).items()}
with open(FAISS_TO_MAL_PATH, "r", encoding="utf-8") as f:
    faiss_id_to_mal_id = {int(k): int(v) for k, v in json.load(f).items()}

# Reconstruct all embeddings once (so we can use them as query vectors)
embeddings = np.vstack([index.reconstruct(i) for i in range(index.ntotal)]).astype("float32")
print("index size:", index.ntotal, "  dim:", embeddings.shape[1])

index size: 16206   dim: 256


In [4]:
# Minimal anime metadata for genre/studio comparison
anime_df = pd.read_csv(DATA_DIR / "anime.csv", usecols=["MAL_ID", "Name", "Genres", "Studios"])
anime_df = anime_df.replace(r"(?i)^unknown$", np.nan, regex=True)
anime_df = anime_df[anime_df["MAL_ID"].isin(mal_id_to_faiss_id)].reset_index(drop=True)

def to_set(s):
    if pd.isna(s):
        return set()
    return {tok.strip() for tok in str(s).split(",") if tok.strip()}

mal_to_genres  = {int(r.MAL_ID): to_set(r.Genres)  for r in anime_df.itertuples(index=False)}
mal_to_studios = {int(r.MAL_ID): to_set(r.Studios) for r in anime_df.itertuples(index=False)}
len(mal_to_genres)

16206

## 1. Genre / Studio Jaccard overlap

For a sample of anime, fetch top-K content neighbors and measure:
- **Genre Jaccard** = |G_query AND G_neighbor| / |G_query OR G_neighbor|
- **Studio match rate** = fraction of neighbors sharing at least one studio with the query

Compare against a **random baseline** (K randomly sampled anime). If the model is doing anything useful, Jaccard should be substantially higher than random.

In [5]:
def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

def neighbors_for(faiss_id: int, k: int):
    """Return list of MAL_IDs for top-k neighbors, excluding the query itself."""
    q = embeddings[faiss_id:faiss_id + 1]
    _, idxs = index.search(q, k + 1)
    out = [faiss_id_to_mal_id[int(i)] for i in idxs[0] if int(i) != faiss_id]
    return out[:k]

In [6]:
K = 10
SAMPLE_SIZE = 2000
rng = np.random.default_rng(42)

all_mal_ids = np.array(list(mal_id_to_faiss_id.keys()))
# Only sample anime that actually have genres recorded, else Jaccard is meaningless
eligible = np.array([m for m in all_mal_ids if mal_to_genres.get(m)])
sample_mal_ids = rng.choice(eligible, size=min(SAMPLE_SIZE, len(eligible)), replace=False)

model_genre_jac, model_studio_hit = [], []
rand_genre_jac,  rand_studio_hit  = [], []

for mal_id in sample_mal_ids:
    q_genres  = mal_to_genres[int(mal_id)]
    q_studios = mal_to_studios.get(int(mal_id), set())
    fid = mal_id_to_faiss_id[int(mal_id)]

    # Model neighbors
    nbrs = neighbors_for(fid, K)
    for n_mal in nbrs:
        model_genre_jac.append(jaccard(q_genres, mal_to_genres.get(n_mal, set())))
        model_studio_hit.append(1.0 if (q_studios and (q_studios & mal_to_studios.get(n_mal, set()))) else 0.0)

    # Random baseline neighbors (same K, exclude self)
    rand_pool = rng.choice(all_mal_ids, size=K + 1, replace=False)
    rand_pool = [m for m in rand_pool if m != mal_id][:K]
    for n_mal in rand_pool:
        rand_genre_jac.append(jaccard(q_genres, mal_to_genres.get(int(n_mal), set())))
        rand_studio_hit.append(1.0 if (q_studios and (q_studios & mal_to_studios.get(int(n_mal), set()))) else 0.0)

results = pd.DataFrame({
    "metric": ["Genre Jaccard@K", "Studio match rate@K"],
    "model":  [np.mean(model_genre_jac), np.mean(model_studio_hit)],
    "random": [np.mean(rand_genre_jac),  np.mean(rand_studio_hit)],
})
results["lift"] = results["model"] / results["random"].replace(0, np.nan)
results

,metric,model,random,lift
0,Genre Jaccard@K,0.648673,0.092976,6.976810
1,Studio match rate@K,0.149850,0.006100,24.565574


## 2. Held-out user ratings as relevance signal

**Setup:** For each sampled user, take their set of anime rated >=8 (call it H). For each anime `a` in H, use `a` as the query and treat `H \ {a}` as the ground-truth relevant set. Get the model's top-K content neighbors of `a` and score:

- **Precision@K** = (# neighbors in H \ {a}) / K
- **Recall@K**    = (# neighbors in H \ {a}) / |H \ {a}|
- **AP@K**        = mean of precision-at-each-relevant-hit

Final reported numbers are means over all (user, query) pairs. **MAP@K** is the mean of the per-query AP@K.

In [7]:
ratings_df = pd.read_csv(
    DATA_DIR / "rating_complete.csv",
    dtype={"user_id": np.int32, "anime_id": np.int32, "rating": np.int8},
)
ratings_df = ratings_df[ratings_df["anime_id"].isin(mal_id_to_faiss_id)]
high = ratings_df[ratings_df["rating"] >= 8]
print("high-rating rows:", len(high))

high-rating rows: 30875402


In [8]:
MIN_HIGH = 5    # need at least this many >=8 ratings to be evaluable
MAX_HIGH = 200  # cap so power-users don't dominate
N_USERS  = 2000

user_high_counts = high.groupby("user_id").size()
eligible_users = user_high_counts[(user_high_counts >= MIN_HIGH) & (user_high_counts <= MAX_HIGH)].index.to_numpy()
print("eligible users:", len(eligible_users))

rng = np.random.default_rng(42)
sampled_users = rng.choice(eligible_users, size=min(N_USERS, len(eligible_users)), replace=False)

user_to_high = (
    high[high["user_id"].isin(sampled_users)]
    .groupby("user_id")["anime_id"]
    .apply(lambda s: set(map(int, s)))
    .to_dict()
)
len(user_to_high)

eligible users: 250169


2000

In [9]:
# Batch-query all unique anime that will be used as queries (faster than per-query search)
K = 10

query_mal_ids = sorted({a for h in user_to_high.values() for a in h})
query_faiss_ids = np.array([mal_id_to_faiss_id[m] for m in query_mal_ids], dtype=np.int64)
query_vecs = embeddings[query_faiss_ids]

_, batch_idxs = index.search(query_vecs, K + 1)

neighbors_map = {}
for mal_id, fid, row in zip(query_mal_ids, query_faiss_ids, batch_idxs):
    nbrs = [faiss_id_to_mal_id[int(i)] for i in row if int(i) != int(fid)][:K]
    neighbors_map[mal_id] = nbrs
len(neighbors_map)

5348

In [10]:
def average_precision_at_k(predicted, relevant_set, k):
    if not relevant_set:
        return 0.0
    hits = 0
    score = 0.0
    for i, p in enumerate(predicted[:k], start=1):
        if p in relevant_set:
            hits += 1
            score += hits / i
    if hits == 0:
        return 0.0
    return score / min(k, len(relevant_set))

precisions, recalls, aps = [], [], []

for user_id, h_set in user_to_high.items():
    if len(h_set) < 2:
        continue
    for a in h_set:
        relevant = h_set - {a}
        preds = neighbors_map.get(a, [])
        if not preds:
            continue
        hits = sum(1 for p in preds if p in relevant)
        precisions.append(hits / K)
        recalls.append(hits / len(relevant))
        aps.append(average_precision_at_k(preds, relevant, K))

summary = pd.DataFrame({
    "metric": [f"Precision@{K}", f"Recall@{K}", f"MAP@{K}"],
    "value":  [np.mean(precisions), np.mean(recalls), np.mean(aps)],
    "n_queries": [len(precisions)] * 3,
})
summary

,metric,value,n_queries
0,Precision@10,0.127539,140991
1,Recall@10,0.015129,140991
2,MAP@10,0.083724,140991


In [11]:
# Random baseline: for the same queries, pick K random anime instead of neighbors
rng = np.random.default_rng(0)
all_ids_arr = np.array(list(mal_id_to_faiss_id.keys()))

rand_p, rand_r, rand_ap = [], [], []
for user_id, h_set in user_to_high.items():
    if len(h_set) < 2:
        continue
    for a in h_set:
        relevant = h_set - {a}
        preds = rng.choice(all_ids_arr, size=K + 1, replace=False)
        preds = [int(p) for p in preds if int(p) != a][:K]
        hits = sum(1 for p in preds if p in relevant)
        rand_p.append(hits / K)
        rand_r.append(hits / len(relevant))
        rand_ap.append(average_precision_at_k(preds, relevant, K))

pd.DataFrame({
    "metric": [f"Precision@{K}", f"Recall@{K}", f"MAP@{K}"],
    "model":  [np.mean(precisions), np.mean(recalls), np.mean(aps)],
    "random": [np.mean(rand_p),     np.mean(rand_r),  np.mean(rand_ap)],
})

,metric,model,random
0,Precision@10,0.127539,0.006471
1,Recall@10,0.015129,0.000613
2,MAP@10,0.083724,0.001962
